# Comparaison des modèles individuels sur le dataset Tx_Rx

In [ ]:
import numpy as np
from sklearn.model_selection import KFold
import pandas as pd

# Utilitaires (identique aux contenus du git dans src de la branche main)

ANTENNA_NAMES = ["041A9F60", "041A9F61", "0F173B12", "0F173B13",
                 "PORT_1", "PORT_2", "PORT_3", "PORT_4"]
ANTENNA_POSITIONS = {
    "041A9F60": (2.8, 2.2, 3.6),
    "041A9F61": (2.7, 3.8, 3.6),
    "PORT_1":   (5.7, 4.0, 3.9),
    "PORT_2":   (8.5, 3.8, 3.9),
    "PORT_3":   (5.6, 1.7, 3.9),
    "PORT_4":   (8.4, 2.1, 3.9),
    "0F173B12": (11.0, 1.7, 4.1),
    "0F173B13": (11.2, 4.0, 4.3),
}



# === Loader ===

META_COLS = ["EPC", "xyz", "x", "y", "z", "polar", "nearestAnt", "nearestD"]

class TxRxDsLoader:
    """
    Loader for ds_Tx_Rx.csv.
    
    Features columns: max__rssi__* and max__Delta_rssi__*
    Distances columns: <antenna>__NONE  (e.g., PORT_1__NONE, 041A9F60__NONE)
    Missing RSSI values replaced with -130 dBm.
    """
    def __init__(self):
        self.feature_cols:  list[str] = []
        self.antenna_names: list[str] = []

    def load(self, path):
        df = pd.read_csv(path)

        dist_cols = [
            c for c in df.columns
            if c not in META_COLS
            and c.endswith("__NONE")
            and not c.startswith("max__")
        ]
        self.antenna_names = [c.replace("__NONE", "") for c in dist_cols]

        feat_cols = [
            c for c in df.columns
            if c not in META_COLS and c not in dist_cols
        ]
        self.feature_cols = feat_cols

        ds = df.copy()
        ds[feat_cols] = ds[feat_cols].fillna(-130)

        df_clean = ds[feat_cols + ["x", "y", "z"] + dist_cols].reset_index(drop=True)

        X           = df_clean[feat_cols].values.astype(np.float32)
        y_xyz       = df_clean[["x", "y", "z"]].values.astype(np.float32)
        y_distances = df_clean[dist_cols].values.astype(np.float32)

        return X, y_xyz, y_distances, df_clean



# === cross validation === 

def cross_validate(X, y, train_fn, predict_fn, eval_fn, aggregate_fn, k=5, random_state=42):
    """
    Perform k-fold cross-validation and return detailed results per fold and aggregated summary.
    
    Args:
        X: Input features
        y: Target values
        train_fn: Function to train a model ((X_train, y_train) -> model)
        predict_fn: Function to make predictions ((model, X_test) -> y_pred))
        eval_fn: Function to evaluate predictions ((y_test, y_pred, test_index) -> dict[str, float])
        aggregate_fn: Function to aggregate fold results into summary statistics
            ((List[Dict[str, float]]) -> Dict[str, Dict[str, float]])
        k: Number of folds
        random_state: Random seed for reproductibility
    Returns:
        Dict with two keys:
        - "folds": List of dictionaries, each containing metrics for a fold.
        - "summary": Aggregated summary of metrics across all folds.
    """
    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
    
    folds_results = []

    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        model = train_fn(X_train, y_train)
        y_pred = predict_fn(model, X_test)

        fold_result = eval_fn(y_test, y_pred, test_index) # Passes test_index to allow eval_fn to access other data
        folds_results.append(fold_result)
        summary_result = aggregate_fn(folds_results)
        
    return {
        "folds": folds_results,
        "summary": summary_result
    }
    


# === eval_fn and aggregate_fn ===

def eval_fn(y_true, y_pred, test_index):
    errs = np.linalg.norm(y_pred - y_true, axis=1)
    return {
        "mae":    mae_3d(errs),
        "3Derror": rmse_3d(errs),
        "rmse":   rmse_xyz(y_true, y_pred),
        "acc_05": threshold_accuracy(errs, 0.5),
        "acc_1":  threshold_accuracy(errs, 1.0),
        "acc_2":  threshold_accuracy(errs, 2.0),
        "acc_3":  threshold_accuracy(errs, 3.0),
    }

def aggregate_fn(folds_results):
    summary = {}
    for key in folds_results[0]:
        vals = np.array([f[key] for f in folds_results])
        summary[key] = {"mean": float(np.mean(vals)), "std": float(np.std(vals))}
    return summary



# === Métriques ===

def mae_3d(errs: np.ndarray) -> float:
    """
    Calculate the Mean Absolute Error (MAE) from an array of errors.

    This function works for:
    - distance errors between predicted and true antenna distances
    - Euclidean position errors in 3D space

    Args:
        errs: Array of errors (absolute distance errors or Euclidean errors)

    Returns:
        Mean error.
    """
    return float(np.mean(errs))

def rmse_3d(errs: np.ndarray) -> float:
    """Calculate the 3D Root Mean Squared Error (RMSE) between true and predicted positions.

    The 3D RMSE is the square root of the average squared Euclidean distances between each pair
    of true and predicted 3D coordinates.

    Args: 
        errs: Euclidean distance errors for each sample
    Returns:
        Root Mean Squared Error of the Euclidean distances between predicted and true 3D positions
    """
    return np.sqrt(np.mean(errs ** 2))

def rmse_xyz(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Per-axis average RMSE over x, y, z coordinates.

    Equivalent to sklearn's sqrt(mean_squared_error(y_true, y_pred)) on (n, 3) arrays,
    which averages the squared errors over all samples AND all 3 axes before taking sqrt.

    This differs from rmse_3d by a factor of 1/√3:
        rmse_xyz = rmse_3d(errs) / √3

    Args:
        y_true: True positions, shape (n, 3)
        y_pred: Predicted positions, shape (n, 3)
    Returns:
        Per-axis RMSE (in meters). Smaller than rmse_3d for the same predictions.
    """
    diff = np.asarray(y_true) - np.asarray(y_pred)
    return float(np.sqrt(np.mean(diff ** 2)))

def threshold_accuracy(errs: np.ndarray, threshold: float) -> float:
    """Calculate the percentage of predictions with error below a given threshold.

    Args:
        errs: Euclidean distance errors for each sample
    Returns:
        Percentage of predictions with error ≤ threshold
    """
    return (errs <= threshold).mean()*100



# === print_summary ===

def print_summary(label, summary):
    m = summary
    print(
        f"{label:<35} "
        f"RMSE={m['rmse']['mean']:.3f}±{m['rmse']['std']:.3f}  "
        f"MAE={m['mae']['mean']:.3f}±{m['mae']['std']:.3f}  "
        f"3Derror={m['3Derror']['mean']:.3f}±{m['3Derror']['std']:.3f}  "
        f"≤0.5m={m['acc_05']['mean']:.1f}%  "
        f"≤1m={m['acc_1']['mean']:.1f}%  "
        f"≤2m={m['acc_2']['mean']:.1f}%  "
        f"≤3m={m['acc_3']['mean']:.1f}%"
    )

In [ ]:
results_cv = {}


print(f"Loading dataset: {"ds_Tx_Rx.csv"}")
loader = TxRxDsLoader()
X, y_xyz, y_distances, df_clean = loader.load("data/ds_Tx_Rx.csv")

Loading dataset: ds_Tx_Rx.csv


In [3]:
# ExtraTrees MultiOutput (baseline)

import time
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.multioutput import MultiOutputRegressor, RegressorChain

_params = dict(n_estimators=200, random_state=0, n_jobs=-1)

def train_et_mo(X_train, y_train):
    m = MultiOutputRegressor(ExtraTreesRegressor(**_params), n_jobs=-1)
    m.fit(X_train, y_train)
    return m

_t0 = time.time()
results_cv["ET_MO"] = cross_validate(
    X, y_xyz,
    train_fn=train_et_mo,
    predict_fn=lambda m, X: m.predict(X),
    eval_fn=eval_fn,
    aggregate_fn=aggregate_fn,
)
print(f"Temps d'exécution : {time.time() - _t0:.1f}s")
print_summary("ExtraTrees MultiOutput", results_cv["ET_MO"]["summary"])


Temps d'exécution : 124.3s
ExtraTrees MultiOutput              MAE=1.099±0.014  3Derror=1.242±0.020  RMSE=0.717±0.011  ≤0.5m=10.6%  ≤1m=50.1%  ≤2m=93.4%  ≤3m=99.0%


In [5]:
# ExtraTrees direct

import time
from sklearn.ensemble import ExtraTreesRegressor

_params = dict(n_estimators=200, random_state=0, n_jobs=-1)

def train_et_direct(X_train, y_train):
    m = ExtraTreesRegressor(**_params)
    m.fit(X_train, y_train)
    return m

_t0 = time.time()
results_cv["ET_direct"] = cross_validate(
    X, y_xyz,
    train_fn=train_et_direct,
    predict_fn=lambda m, X: m.predict(X),
    eval_fn=eval_fn,
    aggregate_fn=aggregate_fn,
)
print(f"Temps d'exécution : {time.time() - _t0:.1f}s")
print_summary("ExtraTrees", results_cv["ET_direct"]["summary"])


Temps d'exécution : 28.8s
ExtraTrees                          MAE=1.104±0.012  3Derror=1.244±0.018  RMSE=0.718±0.010  ≤0.5m=9.9%  ≤1m=49.3%  ≤2m=93.7%  ≤3m=98.9%


In [6]:
# RandomForest MultiOutput

import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor

_params = dict(n_estimators=200, random_state=0, n_jobs=-1)

def train_rf_mo(X_train, y_train):
    m = MultiOutputRegressor(RandomForestRegressor(**_params), n_jobs=-1)
    m.fit(X_train, y_train)
    return m

_t0 = time.time()
results_cv["RF_MO"] = cross_validate(
    X, y_xyz,
    train_fn=train_rf_mo,
    predict_fn=lambda m, X: m.predict(X),
    eval_fn=eval_fn,
    aggregate_fn=aggregate_fn,
)
print(f"Temps d'exécution : {time.time() - _t0:.1f}s")
print_summary("RandomForest MultiOutput", results_cv["RF_MO"]["summary"])

Temps d'exécution : 307.9s
RandomForest MultiOutput            MAE=1.141±0.023  3Derror=1.299±0.033  RMSE=0.750±0.019  ≤0.5m=10.0%  ≤1m=47.7%  ≤2m=91.4%  ≤3m=98.5%


In [7]:
# RandomForest direct

import time
from sklearn.ensemble import RandomForestRegressor

_params = dict(n_estimators=200, random_state=0, n_jobs=-1)

def train_rf_direct(X_train, y_train):
    m = RandomForestRegressor(**_params)
    m.fit(X_train, y_train)
    return m

_t0 = time.time()
results_cv["RF_direct"] = cross_validate(
    X, y_xyz,
    train_fn=train_rf_direct,
    predict_fn=lambda m, X: m.predict(X),
    eval_fn=eval_fn,
    aggregate_fn=aggregate_fn,
)
print(f"Temps d'exécution : {time.time() - _t0:.1f}s")
print_summary("RF_direct", results_cv["RF_direct"]["summary"])


Temps d'exécution : 96.3s
RF_direct                           MAE=1.144±0.018  3Derror=1.302±0.030  RMSE=0.752±0.017  ≤0.5m=10.2%  ≤1m=47.6%  ≤2m=92.0%  ≤3m=98.6%


In [8]:
# HistGradientBoostingRegressor MultiOutput

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
import time

_params_hgbr = dict(max_iter=300, random_state=0)

def train_hgbr_mo(X_train, y_train):
    m = MultiOutputRegressor(HistGradientBoostingRegressor(**_params_hgbr))
    m.fit(X_train, y_train)
    return m

_t0 = time.time()
results_cv["HGBR_MO"] = cross_validate(
    X, y_xyz,
    train_fn=train_hgbr_mo,
    predict_fn=lambda m, X: m.predict(X),
    eval_fn=eval_fn,
    aggregate_fn=aggregate_fn,
)
print(f"Temps d'exécution HGBR_MO : {time.time() - _t0:.1f}s")
print_summary("HGBR_MO", results_cv["HGBR_MO"]["summary"])

Temps d'exécution HGBR_MO : 194.9s
HGBR_MO                             MAE=1.143±0.012  3Derror=1.296±0.021  RMSE=0.748±0.012  ≤0.5m=10.8%  ≤1m=47.3%  ≤2m=91.9%  ≤3m=98.7%


In [9]:
# XGBoost MultiOutput

import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor

_params_xgb_mo = dict(
    n_estimators=400,
    random_state=0,
    n_jobs=-1,
    objective="reg:squarederror",  # Objectif pour la régression
)

def train_xgb_mo(X_train, y_train):
    m = MultiOutputRegressor(xgb.XGBRegressor(**_params_xgb_mo))
    m.fit(X_train, y_train)
    return m

_t0 = time.time()
results_cv["XGB_MO"] = cross_validate(
    X, y_xyz,
    train_fn=train_xgb_mo,
    predict_fn=lambda m, X: m.predict(X),
    eval_fn=eval_fn,
    aggregate_fn=aggregate_fn,
)
print(f"Temps d'exécution XGB_MO : {time.time() - _t0:.1f}s")
print_summary("XGB_MO", results_cv["XGB_MO"]["summary"])

Temps d'exécution XGB_MO : 99.2s
XGB_MO                              MAE=1.225±0.021  3Derror=1.395±0.026  RMSE=0.805±0.015  ≤0.5m=9.2%  ≤1m=42.6%  ≤2m=88.9%  ≤3m=98.0%


In [10]:
# XGBoost direct

import xgboost as xgb

_params_xgb = dict(n_estimators=300, random_state=0, n_jobs=-1)

def train_xgb_direct(X_train, y_train):
    m = xgb.XGBRegressor(objective="reg:squarederror", **_params_xgb)
    m.fit(X_train, y_train)
    return m

_t0 = time.time()
results_cv["XGB_direct"] = cross_validate(
    X, y_xyz,
    train_fn=train_xgb_direct,
    predict_fn=lambda m, X: m.predict(X),
    eval_fn=eval_fn,
    aggregate_fn=aggregate_fn,
)
print(f"Temps d'exécution XGB_direct : {time.time() - _t0:.1f}s")
print_summary("XGB_direct", results_cv["XGB_direct"]["summary"])

Temps d'exécution XGB_direct : 65.0s
XGB_direct                          MAE=1.225±0.021  3Derror=1.394±0.026  RMSE=0.805±0.015  ≤0.5m=9.2%  ≤1m=42.7%  ≤2m=88.9%  ≤3m=98.0%


In [11]:
# CatBoost MultiOutput

from catboost import CatBoostRegressor
from sklearn.multioutput import MultiOutputRegressor

_params_catboost_mo = dict(iterations=300, random_state=0, thread_count=-1, verbose=0)

def train_catboost_mo(X_train, y_train):
    m = MultiOutputRegressor(CatBoostRegressor(**_params_catboost_mo))
    m.fit(X_train, y_train)
    return m

_t0 = time.time()
results_cv["CatBoost_MO"] = cross_validate(
    X, y_xyz,
    train_fn=train_catboost_mo,
    predict_fn=lambda m, X: m.predict(X),
    eval_fn=eval_fn,
    aggregate_fn=aggregate_fn,
)
print(f"Temps d'exécution CatBoost_MO : {time.time() - _t0:.1f}s")
print_summary("CatBoost_MO", results_cv["CatBoost_MO"]["summary"])

Temps d'exécution CatBoost_MO : 204.9s
CatBoost_MO                         MAE=1.161±0.016  3Derror=1.309±0.023  RMSE=0.756±0.013  ≤0.5m=9.7%  ≤1m=45.1%  ≤2m=91.2%  ≤3m=98.7%


In [12]:
# LightGBM MultiOutput

from sklearn.multioutput import MultiOutputRegressor
import lightgbm as lgb

_params_lgbm_mo = dict(
    n_estimators=300,
    random_state=0,
    n_jobs=-1,
    verbose=0
)

def train_lgbm_mo(X_train, y_train):
    m = MultiOutputRegressor(lgb.LGBMRegressor(**_params_lgbm_mo))
    m.fit(X_train, y_train)
    return m

_t0 = time.time()
results_cv["LGBM_MO"] = cross_validate(
    X, y_xyz,
    train_fn=train_lgbm_mo,
    predict_fn=lambda m, X: m.predict(X),
    eval_fn=eval_fn,
    aggregate_fn=aggregate_fn,
)
print(f"Temps d'exécution LGBM_MO : {time.time() - _t0:.1f}s")
print_summary("LGBM_MO", results_cv["LGBM_MO"]["summary"])

c:\Users\lubin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\lubin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\lubin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\lubin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\lubin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserW

Temps d'exécution LGBM_MO : 80.1s
LGBM_MO                             MAE=1.144±0.010  3Derror=1.301±0.017  RMSE=0.751±0.010  ≤0.5m=10.8%  ≤1m=47.3%  ≤2m=91.5%  ≤3m=98.8%


c:\Users\lubin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\lubin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\lubin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [16]:
# TABLEAU RECAPITULATIF

SEP = "=" * 130
print(f"\n\n{SEP}")
print("RECAPITULATIF (cross-validation 5 folds)")
print(SEP)
header = (
    f"{'Approche':<40} "
    f"{'RMSE':>12} "
    f"{'MAE':>12} "
    f"{'3Derror':>12} "
    f"{'≤0.5m':>8} "
    f"{'≤1m':>8} "
    f"{'≤2m':>8} "
    f"{'≤3m':>8}"
)
print(header)
print("-" * 130)

# Trier les résultats par RMSE (moyenne, croissant)
sorted_results = sorted(
    results_cv.items(),
    key=lambda x: x[1]["summary"]["rmse"]["mean"]
)

for label, res in sorted_results:
    s = res["summary"]
    print(
        f"{label:<40} "
        f"{s['rmse']['mean']:>6.3f}±{s['rmse']['std']:.3f}  "
        f"{s['mae']['mean']:>6.3f}±{s['mae']['std']:.3f}  "
        f"{s['3Derror']['mean']:>6.3f}±{s['3Derror']['std']:.3f}  "
        f"{s['acc_05']['mean']:>5.1f}%  "
        f"{s['acc_1']['mean']:>5.1f}%  "
        f"{s['acc_2']['mean']:>5.1f}%  "
        f"{s['acc_3']['mean']:>5.1f}%"
    )
print(SEP)



RECAPITULATIF (cross-validation 5 folds)
Approche                                         RMSE          MAE      3Derror    ≤0.5m      ≤1m      ≤2m      ≤3m
----------------------------------------------------------------------------------------------------------------------------------
ET_MO                                     0.717±0.011   1.099±0.014   1.242±0.020   10.6%   50.1%   93.4%   99.0%
ET_direct                                 0.718±0.010   1.104±0.012   1.244±0.018    9.9%   49.3%   93.7%   98.9%
HGBR_MO                                   0.748±0.012   1.143±0.012   1.296±0.021   10.8%   47.3%   91.9%   98.7%
RF_MO                                     0.750±0.019   1.141±0.023   1.299±0.033   10.0%   47.7%   91.4%   98.5%
LGBM_MO                                   0.751±0.010   1.144±0.010   1.301±0.017   10.8%   47.3%   91.5%   98.8%
RF_direct                                 0.752±0.017   1.144±0.018   1.302±0.030   10.2%   47.6%   92.0%   98.6%
CatBoost_MO               